In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint



# =================================================================================
# PARTE 1: Carregamento dos dados e pré-processamento
# =================================================================================

# --- Carregando o conjunto de dados MNIST no Keras ---
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

digits = test_images.copy() # Criando a variável digits para a função compare_prediction

# --- Covertendo os tensores do MNIST em matrizes e normalizando os valores na escala 0-250 ---
train_images = train_images.reshape((60000, 28*28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28*28))
test_images = test_images.astype("float32") / 255


# =================================================================================
# PARTE 2: Treinamento do modelo baseado em uma rede neural densamente conectada
# =================================================================================

import keras
from keras import layers

# --- Parâmetros do treinamento ---
batch_size = 256
epochs = 480

# --- Arquitetura da rede ---
model = keras.Sequential(
    [
      layers.Dense(1024, activation="relu"),
      layers.Dropout(0.6),
      layers.Dense(512, activation="relu"),
      layers.Dropout(0.4),
      layers.Dense(256, activation="relu"),
      layers.Dense(10, activation="softmax"),
    ]
)

# --- Técnicas de regularização ---
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=40, # se não houver redução da função loss no conjunto de validação após 40 épocas, o treinamento é interrompido
    restore_best_weights=True # Restaura os pesos para àqueles encontrados na época que atingiu o menor val_loss
)

model_checkpoint = ModelCheckpoint(
    filepath=f"mnist_dense_b{batch_size}e{epochs}_last.keras", # Salvando o modelo 
    monitor="val_loss",
    save_best_only=True, # Se o treinamento for executado para todas as épocas, restaura os pesos da época com menor val_loss
)

# --- Compilando o modelo ---
optimizer = Adam(learning_rate=0.0005) # Especificando a taxa de aprendizado
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy", # Função de perda para mais de dois labels
    metrics=["accuracy"], # Métrica para avaliar as classificações do conjunto de treinamento e validação
)

# --- Iniciando o treinamento ---
history = model.fit(
    train_images,
    train_labels,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2, # Separando 20% dos dados do treinamento para a validação do modelo
    callbacks = [model_checkpoint, early_stop] # -> early_stop: opcional caso queira interromper o treinamento
)


# --- Carregando o modelo salvo em model_checkpoint ---
model = keras.models.load_model(f"mnist_dense_b{batch_size}e{epochs}_last.keras")


# ===================================================================================
# PARTE 3: Avaliação do modelo
# ===================================================================================

# --- Avaliando o modelo com os dados em test_images ---
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"test_acc: {test_acc}")

# --- Gráficos  ---
history_dict = history.history # Acessando a perda e acurácia armazenadas no objeto history

# Plotando o gráfico da perda
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "r--", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("[MNIST] Training and validation loss")
plt.xlabel("Epochs")
plt.xticks()
plt.ylabel("Loss")
plt.legend()
plt.show()

# Plotando o gráfico da acurácia
loss_values = history_dict["accuracy"]
val_loss_values = history_dict["val_accuracy"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "r--", label="Training accuracy")
plt.plot(epochs, val_loss_values, "b", label="Validation accuracy")
plt.title("[MNIST] Training and validation accuracy")
plt.xlabel("Epochs")
plt.xticks()
plt.ylabel("Accuracy")
plt.legend()
plt.show()


import random

def compare_prediction(idx):
    image_predict = test_images[idx]
    prediction = model.predict(image_predict[np.newaxis, :])
    predicted_digit = np.argmax(prediction)
    actual_digit = test_labels[idx]

    print(f"Model Prediction: {predicted_digit}")
    print(f"Actual Label: {actual_digit}")

    digit = digits[idx]
    plt.imshow(digit, cmap=plt.cm.binary)
    plt.show()

digit = random.randint(0, 9999)
compare_prediction(digit)


#### Porque converter os dados para float32?

- Os dados de imagem são normalmente armazenados como uint8 (inteiros de 0 a 255). A divisão desses inteiros por 255, para normalizá-los, obtêm decimais entre 0 e 1. Se os dados fossem mantidos como inteiros, qualquer resultado menor do que seria simplesmente arredondado para 0. A conversão para float32, permite que o computador armazene os gradientes sutis e os valores fracionários necessários para o aprendizado do modelo.

- A maioria das bibliotecas de deep learning (como TensorFlow, Keras e PyTorch) são projetadas para trabalhar com números de ponto flutuante.

- Se os números de ponto flutuante são mais precisos, por que não usar float64?

    - Aceleração por hardware: GPUs e TPUs modernas são otimizadas fisicamente para realizar cálculos com números de ponto flutuante de 32 bits muito mais rapidamente do que com números de 64 bits.

    - Uso de memória: Um float64 ocupa o dobro do espaço de um float32. Ao lidar com 60.000 imagens, dobrar o uso de memória pode tornar o treinamento mais lento ou até mesmo travar o sistema, sem proporcionar um aumento significativo na precisão do reconhecimento de imagens.

#### O quê é Dropout?

- O Dropout, desenvolvido por Geoff Hinton e seus alunos na Universidade de Toronto, é uma das técnicas de regularização mais eficazes e comumente usadas para redes neurais. 
   
    - O Dropout, aplicado a uma camada, consiste em descartar aleatoriamente (zerar) um número de características de saída da camada durante o treinamento. Digamos que uma determinada camada normalmente retorne um vetor $[0.2, 0.5, 1.3, 0.8, 1.1]$ para uma determinada amostra de entrada durante o treinamento. Após a aplicação do Dropout, esse vetor terá algumas entradas zero distribuídas aleatoriamente: por exemplo, $[0, 0.5, 1.3, 0, 1.1]$. A taxa de Dropout é a fração das características que são zeradas; geralmente é definida entre 0.2 e 0.5. 
   
    - No momento do teste, nenhuma unidade é descartada; em vez disso, os valores de saída da camada são reduzidos por um fator igual à taxa de Dropout, para compensar o fato de que mais unidades estão ativas do que no momento do treinamento.

    - Essa técnica pode parecer estranha e arbitrária. Por que isso ajudaria a reduzir o sobreajuste? Hinton diz que se inspirou, entre outras coisas, em um mecanismo de prevenção de fraudes usado por bancos:

   >"Fui ao meu banco. Os caixas mudavam constantemente e perguntei a um deles por quê. Ele disse que não sabia, mas que eles eram transferidos com frequência. Imaginei que devia ser porque seria necessária a cooperação entre os funcionários para fraudar o banco com sucesso. Isso me fez perceber que remover aleatoriamente um subconjunto diferente de neurônios em cada exemplo impediria conspirações e, portanto, reduziria o sobreajuste."


  - A ideia central é que introduzir ruído nos valores de saída de uma camada pode quebrar padrões aleatórios que não são significativos (o que Hinton chama de conspirações), que o modelo começará a memorizar se não houver ruído presente.